In [ ]:
# Install dependencies
!apt-get update
!apt-get install -y build-essential cmake libopenblas-dev liblapack-dev libeigen3-dev libopencv-dev libboost-all-dev

# Clone OpenFace and download models
!git clone --depth 1 https://github.com/TadasBaltrusaitis/OpenFace.git
!cd OpenFace && bash download_models.sh

# Clone and build dlib
!git clone https://github.com/davisking/dlib.git
!mkdir -p dlib/build && cd dlib/build && cmake .. && make -j$(nproc) && make install

# Build OpenFace
!cd OpenFace && mkdir -p build && cd build && cmake .. && make -j$(nproc)

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,809 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,266 kB]
Get:13 http://security.ubuntu.com/ubun

In [ ]:
!pip install mediapipe opencv-python pandas numpy scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 13.5 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1

In [ ]:
print("Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

Mounting Google Drive...
Mounted at /content/drive


In [ ]:
import os
import glob
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from scipy.stats import entropy

# Paths
video_dir = "/content/drive/MyDrive/video_clips"
openface_output_dir = "/content/drive/My Drive/openface_output"
mediapipe_output_dir = "/content/drive/My Drive/mediapipe_output"
os.makedirs(openface_output_dir, exist_ok=True)
os.makedirs(mediapipe_output_dir, exist_ok=True)

# AU and landmark collection for all videos
all_au = []
all_landmarks = []
all_combined = []
all_stats = []

# OpenFace AU columns
aus = ['AU01', 'AU06', 'AU12', 'AU14', 'AU25', 'AU26', 'AU45']
au_c = [f'{au}_c' for au in aus]
au_r = [f'{au}_r' for au in aus]

# Feature extraction helpers (from your previous code)
def euclidean(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

FEATURE_PAIRS = {
    "Left Eye Open":      [(398,382), (384,381), (385,380), (386,374), (387,373), (388,390), (466,249)],
    "Right Eye Open":     [(246,7), (161,163), (160,144), (159,145), (158,153), (157,154), (173,155)],
    "Jaw Open":           [(80,170), (81,140), (82,171), (13,175), (312,396), (311,369), (310,395)],
    "Mouth Open":         [(308,78), (81,178), (82,87), (13,14), (312,317), (311,402), (310,318)],
    "Left Eyebrows Raised": [(384,336), (385,296), (386,334), (387,293), (388,300)],
    "Right Eyebrows Raised": [(161,70), (160,63), (159,105), (158,66), (157,107)],
    "Mouth Width":        [(78,308)]
}
FEATURE_NAMES = [
    "Right Eye Open",
    "Left Eye Open",
    "Right Eyebrows Raised",
    "Left Eyebrows Raised",
    "Mouth Open",
    "Mouth Width",
    "Jaw Open"
]
def extract_geometric_features_paper(landmarks):
    mesh = np.array(landmarks)
    mesh_mean = mesh.mean(axis=0)
    mesh_centered = mesh - mesh_mean
    icd = euclidean(mesh_centered[133], mesh_centered[362])
    if icd == 0: icd = 1e-6
    features = {}
    features["Left Eye Open"] = np.mean([euclidean(mesh_centered[a], mesh_centered[b]) for a, b in FEATURE_PAIRS["Left Eye Open"]]) / icd
    features["Right Eye Open"] = np.mean([euclidean(mesh_centered[a], mesh_centered[b]) for a, b in FEATURE_PAIRS["Right Eye Open"]]) / icd
    features["Jaw Open"] = np.mean([euclidean(mesh_centered[a], mesh_centered[b]) for a, b in FEATURE_PAIRS["Jaw Open"]]) / icd
    features["Mouth Open"] = np.mean([euclidean(mesh_centered[a], mesh_centered[b]) for a, b in FEATURE_PAIRS["Mouth Open"]]) / icd
    features["Left Eyebrows Raised"] = np.mean([euclidean(mesh_centered[a], mesh_centered[b]) for a, b in FEATURE_PAIRS["Left Eyebrows Raised"]]) / icd
    features["Right Eyebrows Raised"] = np.mean([euclidean(mesh_centered[a], mesh_centered[b]) for a, b in FEATURE_PAIRS["Right Eyebrows Raised"]]) / icd
    features["Mouth Width"] = euclidean(mesh_centered[78], mesh_centered[308]) / icd
    return features

# Process each video
video_files = sorted(glob.glob(os.path.join(video_dir, "*.mp4")))
for video_path in video_files:
    base = os.path.splitext(os.path.basename(video_path))[0]
    print(f"Processing {base}...")

    # --- 1. Run OpenFace (if not already run) ---
    openface_csv = os.path.join(openface_output_dir, base + '.csv')
    if not os.path.exists(openface_csv):
        !OpenFace/build/bin/FeatureExtraction -f "{video_path}" -out_dir "{openface_output_dir}" -aus

    # --- 2. Extract OpenFace AU features ---
    openface_df = pd.read_csv(openface_csv)
    openface_aus = openface_df[au_c + au_r].reset_index(drop=True)
    openface_aus['clip'] = base
    all_au.append(openface_aus)

    # --- 3. Extract MediaPipe landmarks ---
    mp_face_mesh = mp.solutions.face_mesh
    cap = cv2.VideoCapture(video_path)
    face_mesh = mp_face_mesh.FaceMesh(
        static_image_mode=False,
        max_num_faces=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )
    all_landmarks_clip = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb)
        if results.multi_face_landmarks:
            landmarks = results.multi_face_landmarks[0].landmark
            row = []
            for lm in landmarks:
                row.extend([lm.x, lm.y, lm.z])
            all_landmarks_clip.append(row)
        else:
            all_landmarks_clip.append([np.nan]*468*3)
    cap.release()
    face_mesh.close()
    # Save per-clip landmarks
    landmark_columns = []
    for i in range(468):
        landmark_columns.extend([f"x_{i}", f"y_{i}", f"z_{i}"])
    landmarks_df = pd.DataFrame(all_landmarks_clip, columns=landmark_columns)
    landmarks_df['clip'] = base
    landmarks_df.to_csv(os.path.join(mediapipe_output_dir, f"{base}_landmarks.csv"), index=False)
    all_landmarks.append(landmarks_df)

    # --- 4. Extract geometric features ---
    geometric_features = []
    for row in all_landmarks_clip:
        landmarks = [(row[i*3], row[i*3+1], row[i*3+2]) for i in range(468)]
        if np.isnan(landmarks[0][0]):
            features = {k: np.nan for k in FEATURE_NAMES}
        else:
            features = extract_geometric_features_paper(landmarks)
        geometric_features.append(features)
    geo_df = pd.DataFrame(geometric_features)
    geo_df['clip'] = base

    # --- 5. Combine features ---
    min_len = min(len(geo_df), len(openface_aus))
    combined = pd.concat([geo_df.iloc[:min_len].reset_index(drop=True), openface_aus.iloc[:min_len].reset_index(drop=True)], axis=1)
    all_combined.append(combined)

    # --- 6. Compute stats ---
    stat_features = []
    stat_names = []
    stat_cols = list(geo_df.columns[:-1]) + [f'{au}_r' for au in aus]
    for col in stat_cols:
        vals = combined[col].dropna().values
        stat_features.extend([
            np.mean(vals),
            np.var(vals),
            entropy(np.histogram(vals, bins=10, density=True)[0] + 1e-8)
        ])
        stat_names.extend([f'{col}_mean', f'{col}_var', f'{col}_entropy'])
    stat_row = pd.DataFrame([stat_features], columns=stat_names)
    stat_row['clip'] = base
    all_stats.append(stat_row)

# --- 7. Save all outputs ---
# 1. All_AU.csv
pd.concat(all_au, ignore_index=True).to_csv(os.path.join(openface_output_dir, "All_AU.csv"), index=False)
# 2. All_landmarks.csv
pd.concat(all_landmarks, ignore_index=True).to_csv(os.path.join(mediapipe_output_dir, "All_landmarks.csv"), index=False)
# 3. AU_landmark_combined.csv
pd.concat(all_combined, ignore_index=True).to_csv("AU_landmark_combined.csv", index=False)
# 4. AU_landmark_stat.csv
pd.concat(all_stats, ignore_index=True).to_csv("AU_landmark_stat.csv", index=False)

print("All outputs saved!")

Streaming output truncated to the last 5000 lines.
Reading the PDM module from: OpenFace/build/bin/model/model_eye/pdms/pdm_28_l_eye_3D_closed.txt....Done
Reading the intensity CCNF patch experts from: OpenFace/build/bin/model/model_eye/patch_experts/left_ccnf_patches_1.00_synth_lid_.txt....Done
Reading the intensity CCNF patch experts from: OpenFace/build/bin/model/model_eye/patch_experts/left_ccnf_patches_1.50_synth_lid_.txt....Done
Could not find the HAAR face detector location
Done
Reading part based module....right_eye_28
Reading the landmark detector/tracker from: OpenFace/build/bin/model/model_eye/main_clnf_synth_right.txt
Reading the landmark detector module from: OpenFace/build/bin/model/model_eye/clnf_right_synth.txt
Reading the PDM module from: OpenFace/build/bin/model/model_eye/pdms/pdm_28_eye_3D_closed.txt....Done
Reading the intensity CCNF patch experts from: OpenFace/build/bin/model/model_eye/patch_experts/ccnf_patches_1.00_synth_lid_.txt....Done
Reading the intensity CC

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from scipy.stats import entropy

# Paths
video_dir = "/content/drive/MyDrive/video_clips"
openface_output_dir = "/content/drive/My Drive/openface_output"
mediapipe_output_dir = "/content/drive/My Drive/mediapipe_output"

# AU and landmark collection for all videos
all_combined = []
all_stats = []

# OpenFace AU columns
aus = ['AU01', 'AU06', 'AU12', 'AU14', 'AU25', 'AU26', 'AU45']
au_c = [f'{au}_c' for au in aus]
au_r = [f'{au}_r' for au in aus]

# Feature extraction helpers
def euclidean(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

FEATURE_PAIRS = {
    "Left Eye Open":      [(398,382), (384,381), (385,380), (386,374), (387,373), (388,390), (466,249)],
    "Right Eye Open":     [(246,7), (161,163), (160,144), (159,145), (158,153), (157,154), (173,155)],
    "Jaw Open":           [(80,170), (81,140), (82,171), (13,175), (312,396), (311,369), (310,395)],
    "Mouth Open":         [(308,78), (81,178), (82,87), (13,14), (312,317), (311,402), (310,318)],
    "Left Eyebrows Raised": [(384,336), (385,296), (386,334), (387,293), (388,300)],
    "Right Eyebrows Raised": [(161,70), (160,63), (159,105), (158,66), (157,107)],
    "Mouth Width":        [(78,308)]
}
FEATURE_NAMES = [
    "Right Eye Open",
    "Left Eye Open",
    "Right Eyebrows Raised",
    "Left Eyebrows Raised",
    "Mouth Open",
    "Mouth Width",
    "Jaw Open"
]

def extract_geometric_features_paper(landmarks):
    mesh = np.array(landmarks)
    mesh_mean = mesh.mean(axis=0)
    mesh_centered = mesh - mesh_mean
    icd = euclidean(mesh_centered[133], mesh_centered[362])
    if icd == 0: icd = 1e-8
    features = {}
    features["Left Eye Open"] = np.mean([euclidean(mesh_centered[a], mesh_centered[b]) for a, b in FEATURE_PAIRS["Left Eye Open"]]) / icd
    features["Right Eye Open"] = np.mean([euclidean(mesh_centered[a], mesh_centered[b]) for a, b in FEATURE_PAIRS["Right Eye Open"]]) / icd
    features["Jaw Open"] = np.mean([euclidean(mesh_centered[a], mesh_centered[b]) for a, b in FEATURE_PAIRS["Jaw Open"]]) / icd
    features["Mouth Open"] = np.mean([euclidean(mesh_centered[a], mesh_centered[b]) for a, b in FEATURE_PAIRS["Mouth Open"]]) / icd
    features["Left Eyebrows Raised"] = np.mean([euclidean(mesh_centered[a], mesh_centered[b]) for a, b in FEATURE_PAIRS["Left Eyebrows Raised"]]) / icd
    features["Right Eyebrows Raised"] = np.mean([euclidean(mesh_centered[a], mesh_centered[b]) for a, b in FEATURE_PAIRS["Right Eyebrows Raised"]]) / icd
    features["Mouth Width"] = euclidean(mesh_centered[78], mesh_centered[308]) / icd
    return features

# Process each video
video_files = sorted(glob.glob(os.path.join(video_dir, "*.mp4")))
for video_path in video_files:
    base = os.path.splitext(os.path.basename(video_path))[0]
    print(f"Processing {base}...")

    # --- 1. Load OpenFace AU features ---
    openface_csv = os.path.join(openface_output_dir, base + '.csv')
    openface_df = pd.read_csv(openface_csv)
    # Ensure identifier columns are present
    id_columns = [col for col in ['clip', 'frame', 'timestamp'] if col in openface_df.columns]
    openface_aus = openface_df[(['frame', 'timestamp'] if 'timestamp' in openface_df.columns else ['frame']) + au_c + au_r].reset_index(drop=True)
    openface_aus['clip'] = base

    # --- 2. Load MediaPipe landmarks ---
    landmarks_csv = os.path.join(mediapipe_output_dir, f"{base}_landmarks.csv")
    landmarks_df = pd.read_csv(landmarks_csv)
    # Try to get frame and timestamp if available
    landmark_id_cols = [col for col in ['frame', 'timestamp'] if col in landmarks_df.columns]
    all_landmarks_clip = landmarks_df[[f"x_{i}" for i in range(468)] + [f"y_{i}" for i in range(468)] + [f"z_{i}" for i in range(468)]].values

    # --- 3. Extract geometric features ---
    geometric_features = []
    for idx, row in enumerate(all_landmarks_clip):
        landmarks = [(row[i], row[i+468], row[i+936]) for i in range(468)]
        if np.isnan(landmarks[0][0]):
            features = {k: np.nan for k in FEATURE_NAMES}
        else:
            features = extract_geometric_features_paper(landmarks)
        # Add frame and timestamp if available
        if 'frame' in landmarks_df.columns:
            features['frame'] = landmarks_df.loc[idx, 'frame']
        if 'timestamp' in landmarks_df.columns:
            features['timestamp'] = landmarks_df.loc[idx, 'timestamp']
        features['clip'] = base
        geometric_features.append(features)
    geo_df = pd.DataFrame(geometric_features)

    # --- 4. Combine features ---
    # Reset index before concatenation within the loop
    openface_aus = openface_aus.reset_index(drop=True)
    geo_df = geo_df.reset_index(drop=True)
    min_len = min(len(geo_df), len(openface_aus))
    combined = pd.concat([geo_df.iloc[:min_len], openface_aus.iloc[:min_len]], axis=1)

    # Remove duplicate columns (clip, frame, timestamp)
    for col in ['clip', 'frame', 'timestamp']:
        col_dupes = [c for c in combined.columns if c == col]
        if len(col_dupes) > 1:
            combined = combined.drop(columns=col_dupes[1:])

    # --- 4b. Order columns for AU_landmark_combined.csv ---
    id_columns = ['clip', 'frame', 'timestamp']
    other_columns = [col for col in combined.columns if col not in id_columns]
    combined = combined[[col for col in id_columns if col in combined.columns] + other_columns]

    all_combined.append(combined)

    # --- 5. Compute stats ---
    stat_features = []
    stat_names = []
    stat_cols = [col for col in geo_df.columns if col not in ['clip', 'frame', 'timestamp']] + [f'{au}_r' for au in aus]
    for col in stat_cols:
        vals = combined[col].dropna().values
        stat_features.extend([
            np.mean(vals),
            np.var(vals),
            entropy(np.histogram(vals, bins=10, density=True)[0] + 1e-8)
        ])
        stat_names.extend([f'{col}_mean', f'{col}_var', f'{col}_entropy'])
    stat_row = pd.DataFrame([stat_features], columns=stat_names)
    stat_row['clip'] = base
    all_stats.append(stat_row)

# --- 6. Save outputs with correct column order ---
# 3. AU_landmark_combined.csv
combined_df = pd.concat(all_combined, ignore_index=True)
id_columns = ['clip', 'frame', 'timestamp']
other_columns = [col for col in combined_df.columns if col not in id_columns]
combined_df = combined_df[[col for col in id_columns if col in combined_df.columns] + other_columns]
combined_df.to_csv("AU_landmark_combined.csv", index=False)

# 4. AU_landmark_stat.csv
stats_df = pd.concat(all_stats, ignore_index=True)
stat_columns = [col for col in stats_df.columns if col != 'clip']
stats_df = stats_df[['clip'] + stat_columns]
stats_df.to_csv("AU_landmark_stat.csv", index=False)

print("AU_landmark_combined.csv and AU_landmark_stat.csv saved!")

Processing video0_final...
Processing video100_final...
Processing video101_final...
Processing video102_final...
Processing video103_final...
Processing video104_final...
Processing video105_final...
Processing video106_final...
Processing video107_final...
Processing video108_final...
Processing video10_final...
Processing video110_final...
Processing video111_final...
Processing video112_final...
Processing video113_final...
Processing video117_final...
Processing video118_final...
Processing video119_final...
Processing video11_final...
Processing video120_final...
Processing video121_final...
Processing video122_final...
Processing video123_final...
Processing video125_final...
Processing video12_final...
Processing video130_final...
Processing video132_final...
Processing video133_final...
Processing video134_final...
Processing video135_final...
Processing video136_final...
Processing video137_final...
Processing video138_final...
Processing video139_final...
Processing video13_

In [ ]:
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
from google.colab.patches import cv2_imshow

# Set your video and CSV paths
video_path = '/content/drive/My Drive/video_clips/video0_final.mp4'
landmarks_csv = '/content/drive/My Drive/mediapipe_output/video0_final_landmarks.csv'

# Load landmarks CSV
landmarks_df = pd.read_csv(landmarks_csv)

# Prepare MediaPipe drawing utils
mp_face_mesh = mp.solutions.face_mesh

# Open video
cap = cv2.VideoCapture(video_path)
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret or frame_idx >= len(landmarks_df):
        break

    # Get landmarks for this frame
    row = landmarks_df.iloc[frame_idx]
    h, w = frame.shape[:2]
    landmark_points = []
    for i in range(468):
        x = row[f'x_{i}']
        y = row[f'y_{i}']
        if not np.isnan(x) and not np.isnan(y):
            px = int(x * w)
            py = int(y * h)
            landmark_points.append((px, py))
            # Draw point
            cv2.circle(frame, (px, py), 1, (0,255,0), -1)
        else:
            landmark_points.append(None)

    # # Draw connections
    # for connection in mp_face_mesh.FACE_CONNECTIONS:
    #     start_idx, end_idx = connection
    #     pt1 = landmark_points[start_idx]
    #     pt2 = landmark_points[end_idx]
    #     if pt1 is not None and pt2 is not None:
    #         cv2.line(frame, pt1, pt2, (255,0,0), 1)

    # Show the annotated frame
    cv2_imshow(frame)
    cv2.waitKey(30)

    frame_idx += 1

cap.release()

ModuleNotFoundError: No module named 'mediapipe'

## **Code for 1 clip only**

In [ ]:
from google.colab import files
uploaded = files.upload()
video_path = next(iter(uploaded))

Saving video1.mp4 to video1.mp4


In [ ]:
import os

# Create output directory for OpenFace
openface_out_dir = 'openface_output'
os.makedirs(openface_out_dir, exist_ok=True)

# Run OpenFace FeatureExtraction (assumes OpenFace is built and in the given path)
!OpenFace/build/bin/FeatureExtraction -f "{video_path}" -out_dir "{openface_out_dir}" -aus

Could not find the HAAR face detector location
Reading the landmark detector/tracker from: OpenFace/build/bin/model/main_ceclm_general.txt
Reading the landmark detector module from: OpenFace/build/bin/model/cen_general.txt
Reading the PDM module from: OpenFace/build/bin/model/pdms/In-the-wild_aligned_PDM_68.txt....Done
Reading the Triangulations module from: OpenFace/build/bin/model/tris_68.txt....Done
Reading the intensity CEN patch experts from: OpenFace/build/bin/model/patch_experts/cen_patches_0.25_of.dat....Done
Reading the intensity CEN patch experts from: OpenFace/build/bin/model/patch_experts/cen_patches_0.35_of.dat....Done
Reading the intensity CEN patch experts from: OpenFace/build/bin/model/patch_experts/cen_patches_0.50_of.dat....Done
Reading the intensity CEN patch experts from: OpenFace/build/bin/model/patch_experts/cen_patches_1.00_of.dat....Done
Reading part based module....left_eye_28
Reading the landmark detector/tracker from: OpenFace/build/bin/model/model_eye/main_c

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import os

mediapipe_output_dir = "mediapipe_output"
os.makedirs(mediapipe_output_dir, exist_ok=True)

mp_face_mesh = mp.solutions.face_mesh
cap = cv2.VideoCapture(video_path)
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

all_landmarks = []

while True:
    ret, frame = cap.read()
    if not ret:
        break
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)
    if results.multi_face_landmarks:
        landmarks = results.multi_face_landmarks[0].landmark
        row = []
        for lm in landmarks:
            row.extend([lm.x, lm.y, lm.z])
        all_landmarks.append(row)
    else:
        all_landmarks.append([np.nan]*468*3)

cap.release()
face_mesh.close()

# Save all landmarks as CSV
landmark_columns = []
for i in range(468):
    landmark_columns.extend([f"x_{i}", f"y_{i}", f"z_{i}"])
all_landmarks_df = pd.DataFrame(all_landmarks, columns=landmark_columns)
all_landmarks_df.to_csv(os.path.join(mediapipe_output_dir, "all_landmarks.csv"), index=False)
print("Saved all 468 landmarks per frame to mediapipe_output/all_landmarks.csv")

Saved all 468 landmarks per frame to mediapipe_output/all_landmarks.csv


In [ ]:
import numpy as np

# Helper for Euclidean distance
def euclidean(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

# List of landmark pairs for each feature (from Table 2)
FEATURE_PAIRS = {
    "Left Eye Open":      [(398,382), (384,381), (385,380), (386,374), (387,373), (388,390), (466,249)],
    "Right Eye Open":     [(246,7), (161,163), (160,144), (159,145), (158,153), (157,154), (173,155)],
    "Jaw Open":           [(80,170), (81,140), (82,171), (13,175), (312,396), (311,369), (310,395)],
    "Mouth Open":         [(308,78), (81,178), (82,87), (13,14), (312,317), (311,402), (310,318)],
    "Left Eyebrows Raised": [(384,336), (385,296), (386,334), (387,293), (388,300)],
    "Right Eyebrows Raised": [(161,70), (160,63), (159,105), (158,66), (157,107)],
    "Lip Stretch":        [(323,78), (263,308)],
    "Mouth Width":        [(78,308)]
}

FEATURE_NAMES = [
    "Right Eye Open",
    "Left Eye Open",
    "Right Eyebrows Raised",
    "Left Eyebrows Raised",
    "Mouth Open",
    "Mouth Width",
    "Jaw Open"
]

# For normalization, use InterCanthal Distance (ICD) as in the paper: between 133 and 362
def extract_geometric_features_paper(landmarks):
    # Center the mesh (subtract mean)
    mesh = np.array(landmarks)
    mesh_mean = mesh.mean(axis=0)
    mesh_centered = mesh - mesh_mean

    # InterCanthal Distance (ICD) for normalization
    icd = euclidean(mesh_centered[133], mesh_centered[362])
    if icd == 0: icd = 1e-6

    features = {}
    # Compute each feature as in the paper
    features["Left Eye Open"] = np.mean([euclidean(mesh_centered[a], mesh_centered[b]) for a, b in FEATURE_PAIRS["Left Eye Open"]]) / icd
    features["Right Eye Open"] = np.mean([euclidean(mesh_centered[a], mesh_centered[b]) for a, b in FEATURE_PAIRS["Right Eye Open"]]) / icd
    features["Jaw Open"] = np.mean([euclidean(mesh_centered[a], mesh_centered[b]) for a, b in FEATURE_PAIRS["Jaw Open"]]) / icd
    features["Mouth Open"] = np.mean([euclidean(mesh_centered[a], mesh_centered[b]) for a, b in FEATURE_PAIRS["Mouth Open"]]) / icd
    features["Left Eyebrows Raised"] = np.mean([euclidean(mesh_centered[a], mesh_centered[b]) for a, b in FEATURE_PAIRS["Left Eyebrows Raised"]]) / icd
    features["Right Eyebrows Raised"] = np.mean([euclidean(mesh_centered[a], mesh_centered[b]) for a, b in FEATURE_PAIRS["Right Eyebrows Raised"]]) / icd
    features["Mouth Width"] = euclidean(mesh_centered[78], mesh_centered[308]) / icd
    # Optionally, add Lip Stretch if needed: features["Lip Stretch"] = np.mean([euclidean(mesh_centered[a], mesh_centered[b]) for a, b in FEATURE_PAIRS["Lip Stretch"]]) / icd
    return features

# Usage in your loop:
geometric_features = []
for row in all_landmarks:
    landmarks = [(row[i*3], row[i*3+1], row[i*3+2]) for i in range(468)]
    if np.isnan(landmarks[0][0]):
        features = {k: np.nan for k in FEATURE_NAMES}
    else:
        features = extract_geometric_features_paper(landmarks)
    geometric_features.append(features)
geo_df = pd.DataFrame(geometric_features)
geo_df.to_csv(os.path.join(mediapipe_output_dir, "geometric_features.csv"), index=False)
print("Saved geometric features to mediapipe_output/geometric_features.csv")

Saved geometric features to mediapipe_output/geometric_features.csv


In [ ]:
#Extract OpenFace AU Features
# Find the OpenFace CSV output
openface_output_dir = "openface_output"
base = os.path.splitext(os.path.basename(video_path))[0]
openface_csv = os.path.join(openface_output_dir, base + '.csv')

aus = ['AU01', 'AU06', 'AU12', 'AU14', 'AU25', 'AU26', 'AU45']
au_c = [f'{au}_c' for au in aus]
au_r = [f'{au}_r' for au in aus]
openface_df = pd.read_csv(openface_csv)
openface_aus = openface_df[au_c + au_r].reset_index(drop=True)

In [ ]:
#Combine Features and Save Per-Frame CSV
combined = pd.concat([geo_df, openface_aus], axis=1)
combined.to_csv("per_frame_features.csv", index=False)
print("Per-frame features saved to per_frame_features.csv")

Per-frame features saved to per_frame_features.csv


In [ ]:
#Compute Statistical Aggregates and Save
from scipy.stats import entropy

stat_features = []
stat_names = []
stat_cols = list(geo_df.columns) + [f'{au}_r' for au in aus]
for col in stat_cols:
    vals = combined[col].dropna().values
    stat_features.extend([
        np.mean(vals),
        np.var(vals),
        entropy(np.histogram(vals, bins=10, density=True)[0] + 1e-8)
    ])
    stat_names.extend([f'{col}_mean', f'{col}_var', f'{col}_entropy'])

pd.DataFrame([stat_features], columns=stat_names).to_csv('clip_stats.csv', index=False)
print("Clip-level stats saved to clip_stats.csv")

Clip-level stats saved to clip_stats.csv
